In [ ]:
"""
Fabric Semantic Radar
3. Update the CONFIG section below.
4. Run the notebook.
"""

# %% Cell 1 - Imports

import json
import os
import time
from html import escape
from collections import defaultdict
from datetime import date, datetime
from urllib.parse import quote, urlparse

import msal
import requests
from requests import Response

In [ ]:
# %% Cell 2 - Credential Inputs

# The run preset in Cell 3 sets AUTH_MODE automatically.
# Keep these values only for service principal runs.
AUTH_MODE = "service_principal"

# Service principal settings. Fill these in for service principal mode.For better security use KeyVault instead of putting client secret directly
TENANT_ID = os.getenv("FABRIC_GOVERNANCE_TENANT_ID", "your-tenant-id")
CLIENT_ID = os.getenv("FABRIC_GOVERNANCE_CLIENT_ID", "your-client-id")
CLIENT_SECRET = os.getenv("FABRIC_GOVERNANCE_CLIENT_SECRET", "your-client-secret")

In [ ]:
# %% Cell 3 - Run Preset And Choices

# Pick one preset:
# - "service_principal_capacity": best for full-capacity scans with tenant-approved app access
# - "current_user_capacity_admin": tries admin APIs with the signed-in user, then falls back if needed
# - "current_user_workspace_admin": scans only workspaces the signed-in user can access or explicitly selects
RUN_PRESET = "current_user_workspace_admin"

# Optional workspace targeting for the workspace-admin preset.
# Leave both empty to scan all workspaces the current user can access.
TARGET_WORKSPACE_IDS = []
TARGET_WORKSPACE_NAMES = []

# Preset resolution.
if RUN_PRESET == "service_principal_capacity":
    AUTH_MODE = "service_principal"
    ACCESS_MODE = "capacity_admin"
    USE_ADMIN_APIS = True
    USE_WORKSPACE_SCAN_API = True
elif RUN_PRESET == "current_user_capacity_admin":
    AUTH_MODE = "current_user"
    ACCESS_MODE = "auto"
    USE_ADMIN_APIS = True
    USE_WORKSPACE_SCAN_API = True
elif RUN_PRESET == "current_user_workspace_admin":
    AUTH_MODE = "current_user"
    ACCESS_MODE = "selected_workspaces" if (TARGET_WORKSPACE_IDS or TARGET_WORKSPACE_NAMES) else "auto"
    USE_ADMIN_APIS = False
    USE_WORKSPACE_SCAN_API = False
else:
    raise RuntimeError(
        "RUN_PRESET must be 'service_principal_capacity', "
        "'current_user_capacity_admin', or 'current_user_workspace_admin'."
    )

# Output and behavior choices.
ENABLE_DIAGNOSTICS = True
INCLUDE_RESPONSIBILITY = True
SHOW_TABLES = True
SHOW_NETWORK = False
SHOW_EXECUTIVE_SUMMARY = True
DISPLAY_EMAIL_DRAFTS = True
PRINT_FULL_JSON = False
VISUALS_ONLY = True
SHOW_ONLY_ACTIONABLE = True
GENERATE_EMAIL_DRAFTS = True
ACTION_PLAN_MAX_ROWS = 12
SUPPRESS_LIKELY_DEPLOYMENT_CLONES = True
DEPLOYMENT_CLONE_MIN_WORKSPACES = 3

# API handling choices.
ENABLE_RATE_LIMIT_HANDLING = True
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2
DATASOURCE_REQUEST_DELAY_SECONDS = 0.4
ENRICH_SCAN_MODELS_WITH_DIRECT_DATASOURCE_LOOKUP = True
WORKSPACE_SCAN_BATCH_SIZE = 100
SCAN_STATUS_POLL_SECONDS = 3
SCAN_STATUS_TIMEOUT_SECONDS = 600

# Optional filter. Set to a capacity display name or leave as None.
# This is most useful for service principal or capacity-admin-style runs.
CAPACITY_NAME = None

# Optional SemPy enrichment for tables/measures when available in Fabric runtime.
INCLUDE_SEMPY_METADATA = True

# Minimum number of semantic models sharing the same source before calling it a duplicate group.
DUPLICATE_THRESHOLD = 2
PROBABLE_DUPLICATE_SIMILARITY_THRESHOLD = 0.45

# Optional exclusions for intentional duplicates or sandbox/test models.
EXCLUDED_SEMANTIC_MODEL_IDS = []
EXCLUDED_SEMANTIC_MODEL_NAMES = []
EXCLUDED_WORKSPACE_NAMES = []
EXCLUDED_MODEL_WORKSPACE_PAIRS = []

In [ ]:
# %% Cell 4 - Core Functions

# ============================================================================
# AUTHENTICATION
# ============================================================================

def get_access_token(resource: str = "fabric") -> str:
    auth_mode = AUTH_MODE.strip().lower()

    if auth_mode == "service_principal":
        settings = {
            "TENANT_ID": TENANT_ID,
            "CLIENT_ID": CLIENT_ID,
            "CLIENT_SECRET": CLIENT_SECRET,
        }
        missing = [
            name
            for name, value in settings.items()
            if not value or str(value).startswith("<")
        ]
        if missing:
            raise RuntimeError(
                "Missing service principal configuration in the notebook config section. "
                f"Fill these values: {', '.join(missing)}"
            )

        scopes = {
            "fabric": ["https://api.fabric.microsoft.com/.default"],
            "powerbi": ["https://analysis.windows.net/powerbi/api/.default"],
        }
        app = msal.ConfidentialClientApplication(
            client_id=CLIENT_ID,
            authority=f"https://login.microsoftonline.com/{TENANT_ID}",
            client_credential=CLIENT_SECRET,
        )
        result = app.acquire_token_for_client(scopes=scopes[resource])
        token = result.get("access_token")
        if not token:
            error = result.get("error_description") or result.get("error") or "Unknown MSAL error"
            raise RuntimeError(f"Could not acquire {resource} token via service principal: {error}")
        return token

    if auth_mode in {"notebook", "current_user"}:
        from notebookutils import credentials  # type: ignore

        token = credentials.getToken("pbi")
        if not token:
            raise RuntimeError("Could not acquire a Fabric/Power BI access token from notebookutils.")
        return token

    raise RuntimeError("AUTH_MODE must be 'service_principal', 'current_user', or 'notebook'.")


def headers(access_token: str) -> dict[str, str]:
    return {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }


def to_json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}

    if isinstance(value, (list, tuple, set)):
        return [to_json_safe(item) for item in value]

    isoformat = getattr(value, "isoformat", None)
    if callable(isoformat):
        try:
            return isoformat()
        except Exception:
            pass

    tolist = getattr(value, "tolist", None)
    if callable(tolist):
        try:
            return to_json_safe(tolist())
        except Exception:
            pass

    return str(value)


def truncate_text(value: str | None, max_length: int = 120) -> str:
    text = str(value or "").strip()
    if len(text) <= max_length:
        return text
    return text[: max_length - 3].rstrip() + "..."


def request_with_retry(
    method: str,
    url: str,
    *,
    headers_dict: dict,
    params: dict | None = None,
    timeout: int = 60,
) -> Response:
    last_response = None

    for attempt in range(MAX_RETRIES + 1):
        response = requests.request(
            method,
            url,
            headers=headers_dict,
            params=params,
            timeout=timeout,
        )

        if response.status_code != 429:
            response.raise_for_status()
            return response

        last_response = response
        if not ENABLE_RATE_LIMIT_HANDLING or attempt >= MAX_RETRIES:
            response.raise_for_status()

        retry_after_header = response.headers.get("Retry-After")
        if retry_after_header:
            try:
                sleep_seconds = max(int(retry_after_header), 1)
            except ValueError:
                sleep_seconds = RETRY_BASE_SECONDS * (2 ** attempt)
        else:
            sleep_seconds = RETRY_BASE_SECONDS * (2 ** attempt)

        print(
            f"Rate limited on {url}. "
            f"Retrying in {sleep_seconds} seconds (attempt {attempt + 1}/{MAX_RETRIES})."
        )
        time.sleep(sleep_seconds)

    if last_response is not None:
        last_response.raise_for_status()
    raise RuntimeError(f"Request failed after retries: {url}")


# ============================================================================
# HELPERS
# ============================================================================

def get_paginated(url: str, access_token: str, value_key: str) -> list[dict]:
    items = []
    continuation_token = None

    while True:
        response = request_with_retry(
            "GET",
            url,
            headers_dict=headers(access_token),
            params={"continuationToken": continuation_token} if continuation_token else None,
            timeout=60,
        )
        payload = response.json()
        items.extend(payload.get(value_key, []))

        continuation_token = payload.get("continuationToken")
        if not continuation_token:
            break

    return items


def get_paginated_with_params(url: str, access_token: str, value_key: str, params: dict | None = None) -> list[dict]:
    items = []
    continuation_token = None

    while True:
        request_params = dict(params or {})
        if continuation_token:
            request_params["continuationToken"] = continuation_token

        response = request_with_retry(
            "GET",
            url,
            headers_dict=headers(access_token),
            params=request_params or None,
            timeout=60,
        )
        payload = response.json()
        items.extend(payload.get(value_key, []))

        continuation_token = payload.get("continuationToken")
        if not continuation_token:
            break

    return items


def normalize_datasource(datasource_type: str, connection_details: dict) -> tuple[str, dict]:
    normalized = {}
    for key, value in sorted(connection_details.items()):
        if value is None:
            continue

        text = str(value).strip().lower()
        if not text:
            continue

        normalized_key = str(key).strip().lower()
        if normalized_key == "url":
            parsed = urlparse(text)
            text = f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip("/")

        normalized[normalized_key] = text

    fingerprint = (
        f"{str(datasource_type).strip().lower()}:"
        f"{json.dumps(normalized, sort_keys=True, separators=(',', ':'))}"
    )
    return fingerprint, normalized


def try_get_semantic_model_metadata(dataset_name: str, workspace_name: str | None = None) -> dict:
    try:
        import sempy.fabric as fabric  # type: ignore
    except ImportError:
        return {}

    metadata = {}

    try:
        tables = fabric.list_tables(dataset_name, workspace=workspace_name)
        metadata["table_count"] = len(tables)
        metadata["tables"] = tables.to_dict(orient="records")
    except Exception:
        pass

    try:
        measures = fabric.list_measures(dataset_name, workspace=workspace_name)
        metadata["measure_count"] = len(measures)
        metadata["measures"] = measures.to_dict(orient="records")
    except Exception:
        pass

    try:
        columns = fabric.list_columns(dataset_name, workspace=workspace_name)
        metadata["column_count"] = len(columns)
        metadata["columns"] = columns.to_dict(orient="records")
    except Exception:
        pass

    return metadata


def list_admin_datasets(powerbi_token: str, top: int = 5000) -> list[dict]:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    items = []
    skip = 0

    while True:
        response = request_with_retry(
            "GET",
            f"{power_bi_base_url}/admin/datasets",
            headers_dict=headers(powerbi_token),
            params={"$top": top, "$skip": skip},
            timeout=60,
        )
        page = response.json().get("value", [])
        items.extend(page)
        if len(page) < top:
            break
        skip += top

    return items


def list_admin_capacities(powerbi_token: str) -> list[dict]:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    response = request_with_retry(
        "GET",
        f"{power_bi_base_url}/admin/capacities",
        headers_dict=headers(powerbi_token),
        timeout=60,
    )
    return response.json().get("value", [])


def list_admin_workspaces(fabric_token: str, capacity_id: str | None = None) -> list[dict]:
    fabric_base_url = "https://api.fabric.microsoft.com/v1"
    params = {"type": "Workspace"}
    if capacity_id:
        params["capacityId"] = capacity_id

    return get_paginated_with_params(
        f"{fabric_base_url}/admin/workspaces",
        fabric_token,
        "workspaces",
        params=params,
    )


def list_powerbi_admin_groups(powerbi_token: str, capacity_id: str | None = None, top: int = 5000) -> list[dict]:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    items = []
    skip = 0

    while True:
        response = request_with_retry(
            "GET",
            f"{power_bi_base_url}/admin/groups",
            headers_dict=headers(powerbi_token),
            params={"$top": top, "$skip": skip},
            timeout=60,
        )
        page = response.json().get("value", [])
        if capacity_id:
            page = [item for item in page if item.get("capacityId") == capacity_id]
        items.extend(page)
        if len(response.json().get("value", [])) < top:
            break
        skip += top

    return items


def get_dataset_datasources(powerbi_token: str, workspace_id: str, dataset_id: str, use_admin_apis: bool) -> list[dict]:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"

    if use_admin_apis:
        url = f"{power_bi_base_url}/admin/datasets/{dataset_id}/datasources"
    else:
        url = f"{power_bi_base_url}/groups/{workspace_id}/datasets/{dataset_id}/datasources"

    if DATASOURCE_REQUEST_DELAY_SECONDS > 0:
        time.sleep(DATASOURCE_REQUEST_DELAY_SECONDS)

    response = request_with_retry(
        "GET",
        url,
        headers_dict=headers(powerbi_token),
        timeout=60,
    )
    return response.json().get("value", [])


def get_group_users_as_admin(powerbi_token: str, workspace_id: str) -> list[dict]:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    response = request_with_retry(
        "GET",
        f"{power_bi_base_url}/admin/groups/{workspace_id}/users",
        headers_dict=headers(powerbi_token),
        timeout=60,
    )
    return response.json().get("value", [])


def chunk_list(values: list, size: int) -> list[list]:
    return [values[index:index + size] for index in range(0, len(values), size)]


def normalize_string_set(values: list[str] | tuple[str, ...] | set[str] | None) -> set[str]:
    return {
        str(value).strip().lower()
        for value in (values or [])
        if str(value).strip()
    }


def normalize_model_workspace_pairs(
    values: list[tuple[str, str]] | list[dict] | None,
) -> set[tuple[str, str]]:
    normalized_pairs = set()
    for value in values or []:
        if isinstance(value, dict):
            workspace_name = str(value.get("workspace") or value.get("workspace_name") or "").strip().lower()
            model_name = str(value.get("model") or value.get("display_name") or value.get("name") or "").strip().lower()
        else:
            try:
                workspace_name, model_name = value
            except Exception:
                continue
            workspace_name = str(workspace_name).strip().lower()
            model_name = str(model_name).strip().lower()

        if workspace_name and model_name:
            normalized_pairs.add((workspace_name, model_name))

    return normalized_pairs


def is_excluded_semantic_model(model: dict) -> bool:
    model_id = str(model.get("id") or "").strip()
    model_name = str(model.get("display_name") or model.get("name") or "").strip().lower()
    workspace_name = str(model.get("workspace_name") or "").strip().lower()

    excluded_ids = {str(value).strip() for value in EXCLUDED_SEMANTIC_MODEL_IDS if str(value).strip()}
    excluded_model_names = normalize_string_set(EXCLUDED_SEMANTIC_MODEL_NAMES)
    excluded_workspace_names = normalize_string_set(EXCLUDED_WORKSPACE_NAMES)
    excluded_pairs = normalize_model_workspace_pairs(EXCLUDED_MODEL_WORKSPACE_PAIRS)

    if model_id and model_id in excluded_ids:
        return True
    if model_name and model_name in excluded_model_names:
        return True
    if workspace_name and workspace_name in excluded_workspace_names:
        return True
    if workspace_name and model_name and (workspace_name, model_name) in excluded_pairs:
        return True
    return False


def filter_workspaces_for_user_scope(workspaces: list[dict]) -> list[dict]:
    target_workspace_ids = {str(value).strip() for value in (TARGET_WORKSPACE_IDS or []) if str(value).strip()}
    target_workspace_names = normalize_string_set(TARGET_WORKSPACE_NAMES)

    if ACCESS_MODE == "selected_workspaces":
        return [
            workspace
            for workspace in workspaces
            if workspace.get("id") in target_workspace_ids
            or str(workspace.get("displayName") or workspace.get("name") or "").strip().lower() in target_workspace_names
        ]

    return workspaces


def post_workspace_scan(
    powerbi_token: str,
    workspace_ids: list[str],
    *,
    datasource_details: bool = True,
    dataset_schema: bool = True,
    dataset_expressions: bool = False,
    get_artifact_users: bool = True,
    lineage: bool = True,
) -> dict:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    response = requests.post(
        f"{power_bi_base_url}/admin/workspaces/getInfo",
        headers=headers(powerbi_token),
        params={
            "lineage": str(lineage).lower(),
            "datasourceDetails": str(datasource_details).lower(),
            "datasetSchema": str(dataset_schema).lower(),
            "datasetExpressions": str(dataset_expressions).lower(),
            "getArtifactUsers": str(get_artifact_users).lower(),
        },
        json={"workspaces": workspace_ids},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def get_workspace_scan_status(powerbi_token: str, scan_id: str) -> dict:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    response = request_with_retry(
        "GET",
        f"{power_bi_base_url}/admin/workspaces/scanStatus/{scan_id}",
        headers_dict=headers(powerbi_token),
        timeout=60,
    )
    return response.json()


def get_workspace_scan_result(powerbi_token: str, scan_id: str) -> dict:
    power_bi_base_url = "https://api.powerbi.com/v1.0/myorg"
    response = request_with_retry(
        "GET",
        f"{power_bi_base_url}/admin/workspaces/scanResult/{scan_id}",
        headers_dict=headers(powerbi_token),
        timeout=60,
    )
    return response.json()


def scan_workspaces_metadata(powerbi_token: str, workspace_ids: list[str]) -> dict:
    combined = {
        "datasourceInstances": [],
        "misconfiguredDatasourceInstances": [],
        "workspaces": [],
    }

    for batch in chunk_list(workspace_ids, WORKSPACE_SCAN_BATCH_SIZE):
        scan_request = post_workspace_scan(powerbi_token, batch)
        scan_id = scan_request["id"]
        started = time.time()

        while True:
            status_payload = get_workspace_scan_status(powerbi_token, scan_id)
            status = status_payload.get("status")
            if status == "Succeeded":
                break
            if status == "Failed":
                error = status_payload.get("error", {})
                raise RuntimeError(f"Workspace scan failed for {scan_id}: {error}")
            if (time.time() - started) > SCAN_STATUS_TIMEOUT_SECONDS:
                raise TimeoutError(f"Workspace scan timed out for {scan_id}")
            time.sleep(SCAN_STATUS_POLL_SECONDS)

        scan_result = get_workspace_scan_result(powerbi_token, scan_id)
        combined["datasourceInstances"].extend(scan_result.get("datasourceInstances", []))
        combined["misconfiguredDatasourceInstances"].extend(scan_result.get("misconfiguredDatasourceInstances", []))
        combined["workspaces"].extend(scan_result.get("workspaces", []))

    return combined


def get_scan_linked_datasources(model: dict, datasource_instance_lookup: dict) -> list[dict]:
    usage_ids = []
    for usage_group in ["datasourceUsages", "misconfiguredDatasourceUsages"]:
        for usage in model.get(usage_group, []):
            datasource_instance_id = usage.get("datasourceInstanceId")
            if datasource_instance_id:
                usage_ids.append(datasource_instance_id)

    return [
        datasource_instance_lookup[datasource_instance_id]
        for datasource_instance_id in usage_ids
        if datasource_instance_id in datasource_instance_lookup
    ]


def build_visual_tables(results: dict) -> dict:
    model_lookup = {model["id"]: model for model in results["semantic_models"]}

    consolidation_rows = []
    for group in results["centralization_candidates"]:
        owner_candidates = []
        workspace_admin_candidates = []
        for impacted_model in group["models"]:
            model_detail = model_lookup.get(impacted_model["id"], {})
            owner = model_detail.get("configured_by")
            if owner:
                owner_candidates.append(owner)
            for admin in model_detail.get("workspace_admins", []):
                email = admin.get("emailAddress") or admin.get("identifier")
                if email:
                    workspace_admin_candidates.append(email)

        consolidation_rows.append(
            {
                "priority": next(
                    (
                        recommendation["priority"]
                        for recommendation in results["recommendations"]
                        if recommendation["datasource"] == group["normalized_details"]
                    ),
                    "unknown",
                ),
                "datasource_type": group["datasource_type"],
                "model_count": group["model_count"],
                "workspace_count": len({model["workspace_id"] for model in group["models"]}),
                "models": ", ".join(model["display_name"] for model in group["models"]),
                "workspaces": ", ".join(sorted({model["workspace_name"] for model in group["models"]})),
                "suggested_dataset_owners": ", ".join(sorted(set(owner_candidates))),
                "workspace_admin_contacts": ", ".join(sorted(set(workspace_admin_candidates))),
                "source_details": json.dumps(to_json_safe(group["normalized_details"]), sort_keys=True),
            }
        )

    responsibility_rows = []
    for model in results["semantic_models"]:
        admins = model.get("workspace_admins", [])
        responsibility_rows.append(
            {
                "semantic_model": model["display_name"],
                "workspace": model["workspace_name"],
                "dataset_owner": model.get("configured_by"),
                "workspace_admins": ", ".join(
                    sorted(
                        {
                            admin.get("emailAddress") or admin.get("identifier") or ""
                            for admin in admins
                        }
                    )
                ),
                "datasource_count": len(model.get("data_sources", [])),
                "table_count": model.get("sempy_metadata", {}).get("table_count"),
                "measure_count": model.get("sempy_metadata", {}).get("measure_count"),
            }
        )

    probable_duplicate_rows = []
    for candidate in results.get("probable_duplicate_candidates", []):
        impacted_model_names = candidate.get("impacted_model_names")
        if impacted_model_names is None:
            impacted_model_names = [
                value
                for value in [candidate.get("left_model"), candidate.get("right_model")]
                if value
            ]
        probable_duplicate_rows.append(
            {
                "priority": candidate["priority"],
                "similarity_score": candidate["similarity_score"],
                "model_scope": (
                    candidate.get("model_name")
                    or " vs ".join(impacted_model_names)
                ),
                "model_count": candidate.get("model_count", len(candidate.get("impacted_model_ids", []))),
                "workspace_count": candidate.get("workspace_count", len(candidate.get("impacted_workspaces", []))),
                "workspaces": ", ".join(candidate.get("impacted_workspaces", [])),
                "datasource_visibility": candidate["datasource_visibility"],
                "shared_entities": ", ".join(candidate["shared_entities"]),
                "shared_columns": ", ".join(candidate["shared_columns"]),
                "why": candidate["why"],
            }
        )

    source_replacement_rows = []
    for candidate in results.get("similarity_candidates", []):
        if candidate["recommendation_type"] != "source_replacement_candidate":
            continue
        source_replacement_rows.append(
            {
                "priority": candidate["replacement_confidence"],
                "similarity_score": candidate["similarity_score"],
                "left_model": candidate["left_model"],
                "left_workspace": candidate["left_workspace"],
                "right_model": candidate["right_model"],
                "right_workspace": candidate["right_workspace"],
                "hypothesis": candidate["source_replacement_hypothesis"],
            }
        )

    action_plan_rows = []
    for item in results.get("action_plan", [])[:ACTION_PLAN_MAX_ROWS]:
        workspaces = [value.strip() for value in item.get("workspaces", "").split(",") if value.strip()]
        action_plan_rows.append(
            {
                "rank": item["rank"],
                "priority": item["priority"],
                "action_type": item["action_type"],
                "impacted_model_count": item.get("impacted_model_count"),
                "workspace_count": len(set(workspaces)),
                "workspaces": ", ".join(sorted(set(workspaces))[:6]),
                "next_step": item.get("what_to_do_next"),
                "why": item.get("why"),
            }
        )

    return {
        "consolidation_candidates_table": consolidation_rows,
        "responsibility_table": responsibility_rows,
        "probable_duplicate_table": probable_duplicate_rows,
        "source_replacement_table": source_replacement_rows,
        "action_plan_table": action_plan_rows,
    }


def build_action_plan(results: dict) -> list[dict]:
    impacted_model_ids = set()
    model_lookup = {model["id"]: model for model in results.get("semantic_models", [])}
    plan = []
    suppressed_probable_duplicate_count = 0

    for rank, group in enumerate(results.get("centralization_candidates", []), start=1):
        impacted_ids = [model["id"] for model in group["models"]]
        impacted_model_ids.update(impacted_ids)
        owners = sorted(
            {
                model_lookup[model_id].get("configured_by")
                for model_id in impacted_ids
                if model_lookup.get(model_id, {}).get("configured_by")
            }
        )
        workspace_admins = sorted(
            {
                admin.get("emailAddress") or admin.get("identifier")
                for model_id in impacted_ids
                for admin in model_lookup.get(model_id, {}).get("workspace_admins", [])
                if admin.get("emailAddress") or admin.get("identifier")
            }
        )

        plan.append(
            {
                "rank": rank,
                "action_type": "centralize_duplicate_source",
                "priority": "high" if group["model_count"] >= 4 else "medium",
                "what_to_do_next": (
                    "Design or extend one shared semantic model for this duplicated source, "
                    "move shared entities/measures there, and plan downstream report migration."
                ),
                "why": f"The same source is used by {group['model_count']} semantic models.",
                "impacted_model_count": group["model_count"],
                "impacted_models": ", ".join(model["display_name"] for model in group["models"]),
                "workspaces": ", ".join(sorted({model["workspace_name"] for model in group["models"]})),
                "owners_to_contact": ", ".join(owners),
                "workspace_admins_to_contact": ", ".join(workspace_admins),
                "checklist": "1. confirm overlap 2. pick target shared model 3. migrate measures/tables 4. repoint reports 5. retire duplicates",
            }
        )

    for candidate in results.get("similarity_candidates", []):
        if candidate["recommendation_type"] != "source_replacement_candidate":
            continue

        impacted_model_ids.add(candidate["left_model_id"])
        impacted_model_ids.add(candidate["right_model_id"])
        left_model = model_lookup.get(candidate["left_model_id"], {})
        right_model = model_lookup.get(candidate["right_model_id"], {})
        owners = sorted(
            {
                owner
                for owner in [left_model.get("configured_by"), right_model.get("configured_by")]
                if owner
            }
        )
        workspace_admins = sorted(
            {
                admin.get("emailAddress") or admin.get("identifier")
                for model in [left_model, right_model]
                for admin in model.get("workspace_admins", [])
                if admin.get("emailAddress") or admin.get("identifier")
            }
        )

        plan.append(
            {
                "rank": len(plan) + 1,
                "action_type": "replace_weak_source",
                "priority": candidate["replacement_confidence"],
                "what_to_do_next": (
                    "Validate whether the extract-based model can be replaced by the more authoritative source "
                    "and migrate the reports to the governed semantic layer."
                ),
                "why": candidate["source_replacement_hypothesis"],
                "impacted_model_count": 2,
                "impacted_models": f"{candidate['left_model']}, {candidate['right_model']}",
                "workspaces": f"{candidate['left_workspace']}, {candidate['right_workspace']}",
                "owners_to_contact": ", ".join(owners),
                "workspace_admins_to_contact": ", ".join(workspace_admins),
                "checklist": "1. compare row counts/keys 2. validate refresh timing 3. confirm authoritative source 4. swap reports 5. decommission extract model",
            }
        )

    for candidate in results.get("probable_duplicate_candidates", []):
        if (
            SUPPRESS_LIKELY_DEPLOYMENT_CLONES
            and is_likely_deployment_clone_candidate(candidate)
        ):
            suppressed_probable_duplicate_count += 1
            continue

        impacted_ids = candidate.get("impacted_model_ids")
        if impacted_ids is None:
            impacted_ids = [
                value
                for value in [candidate.get("left_model_id"), candidate.get("right_model_id")]
                if value
            ]
        impacted_model_ids.update(impacted_ids)
        owners = sorted(
            {
                model_lookup[model_id].get("configured_by")
                for model_id in impacted_ids
                if model_lookup.get(model_id, {}).get("configured_by")
            }
        )
        workspace_admins = sorted(
            {
                admin.get("emailAddress") or admin.get("identifier")
                for model_id in impacted_ids
                for admin in model_lookup.get(model_id, {}).get("workspace_admins", [])
                if admin.get("emailAddress") or admin.get("identifier")
            }
        )
        impacted_models = candidate.get("impacted_model_names")
        if impacted_models is None:
            impacted_models = [
                value
                for value in [candidate.get("left_model"), candidate.get("right_model")]
                if value
            ]
        workspaces = candidate.get("impacted_workspaces")
        if workspaces is None:
            workspaces = [
                value
                for value in [candidate.get("left_workspace"), candidate.get("right_workspace")]
                if value
            ]
        probable_checklist = (
            "1. confirm entity overlap 2. compare key tables/measures 3. check intended audience and grain "
            "4. decide merge vs coexist 5. document ownership"
        )
        if len(impacted_ids) > 2:
            probable_checklist = (
                "1. validate the full overlap cluster 2. identify the canonical model "
                "3. compare shared tables/measures across all models 4. decide merge vs coexist "
                "5. document ownership and migration plan"
            )

        plan.append(
            {
                "rank": len(plan) + 1,
                "action_type": "review_probable_duplicate_semantic_model",
                "priority": candidate["priority"],
                "what_to_do_next": (
                    "Review the two semantic models together, compare their business grain and reusable logic, "
                    "and decide whether they should be merged, centralized, or kept intentionally separate."
                ),
                "why": (
                    f"{candidate['why']} Impacted model count: {len(impacted_ids)}."
                    if len(impacted_ids) > 2
                    else candidate["why"]
                ),
                "impacted_model_count": len(impacted_ids),
                "impacted_models": ", ".join(impacted_models),
                "workspaces": ", ".join(workspaces),
                "owners_to_contact": ", ".join(owners),
                "workspace_admins_to_contact": ", ".join(workspace_admins),
                "checklist": probable_checklist,
            }
        )

    results["actionable_model_ids"] = sorted(impacted_model_ids)
    results["suppressed_probable_duplicate_count"] = suppressed_probable_duplicate_count
    return plan


def build_condensed_results(results: dict) -> dict:
    actionable_ids = set(results.get("actionable_model_ids", []))
    actionable_models = [
        {
            "id": model["id"],
            "display_name": model["display_name"],
            "workspace_name": model["workspace_name"],
            "configured_by": model.get("configured_by"),
            "workspace_admins": [
                admin.get("emailAddress") or admin.get("identifier")
                for admin in model.get("workspace_admins", [])
                if admin.get("emailAddress") or admin.get("identifier")
            ],
            "datasource_count": len(model.get("data_sources", [])),
            "table_count": model.get("sempy_metadata", {}).get("table_count"),
            "measure_count": model.get("sempy_metadata", {}).get("measure_count"),
        }
        for model in results.get("semantic_models", [])
        if model["id"] in actionable_ids
    ]

    return {
        "executive_summary": results.get("executive_summary", {}),
        "action_plan": results.get("action_plan", []),
        "action_plan_text": results.get("action_plan_text", ""),
        "email_drafts": results.get("email_drafts", []),
        "actionable_semantic_models": actionable_models,
        "probable_duplicate_candidates": results.get("probable_duplicate_candidates", []),
        "access_issues": results.get("access_issues", []),
        "diagnostics": results.get("diagnostics", {}),
    }


def build_action_plan_text(results: dict) -> str:
    plan = results.get("action_plan", [])
    if not plan:
        return "No immediate consolidation actions were identified."

    lines = ["Action Plan", ""]
    for item in plan:
        lines.append(
            f"{item['rank']}. [{item['priority'].upper()}] {item['action_type']}"
        )
        lines.append(f"Why: {item['why']}")
        lines.append(f"Next: {item['what_to_do_next']}")
        lines.append(f"Models: {item['impacted_models']}")
        lines.append(f"Workspaces: {item['workspaces']}")
        if item.get("owners_to_contact"):
            lines.append(f"Owners: {item['owners_to_contact']}")
        if item.get("workspace_admins_to_contact"):
            lines.append(f"Workspace admins: {item['workspace_admins_to_contact']}")
        lines.append(f"Checklist: {item['checklist']}")
        lines.append("")

    return "\n".join(lines).strip()


def build_email_drafts(results: dict) -> list[dict]:
    if not GENERATE_EMAIL_DRAFTS:
        return []

    drafts = []
    for item in results.get("action_plan", []):
        recipients = []
        for raw_group in [item.get("owners_to_contact", ""), item.get("workspace_admins_to_contact", "")]:
            for value in raw_group.split(","):
                email = value.strip()
                if email:
                    recipients.append(email)

        recipients = sorted(set(recipients))
        if not recipients:
            continue

        subject = f"Semantic model consolidation follow-up: {item['action_type']}"
        body = "\n".join(
            [
                "Hello,",
                "",
                "We reviewed the current semantic model landscape and identified a consolidation opportunity that likely needs your input.",
                "",
                f"Priority: {item['priority']}",
                f"Reason: {item['why']}",
                f"Impacted models: {item['impacted_models']}",
                f"Impacted workspaces: {item['workspaces']}",
                "",
                "Suggested next step:",
                item["what_to_do_next"],
                "",
                "Proposed checklist:",
                item["checklist"],
                "",
                "Would you be open to a short working session to validate the overlap, agree on the target shared semantic model, and plan the migration steps?",
                "",
                "Suggested meeting length: 30 minutes",
                "Suggested outcome: agree on owner, target model, and migration approach",
                "",
                "Best regards,",
                "Semantic Governance Review",
            ]
        )

        drafts.append(
            {
                "rank": item["rank"],
                "action_type": item["action_type"],
                "to": recipients,
                "subject": subject,
                "body": body,
            }
        )

    return drafts


def build_mailto_link(recipients: list[str], subject: str, body: str) -> str:
    query_parts = []
    if subject:
        query_parts.append(f"subject={quote(subject)}")
    if body:
        query_parts.append(f"body={quote(body)}")
    query = "&".join(query_parts)
    return f"mailto:{quote(';'.join(recipients))}" + (f"?{query}" if query else "")


def render_email_drafts_table(email_drafts: list[dict]) -> str:
    rows = [
        "<table>",
        "<thead><tr><th>Rank</th><th>Action</th><th>To</th><th>Subject</th><th>Send</th></tr></thead>",
        "<tbody>",
    ]
    for draft in email_drafts:
        mailto_link = build_mailto_link(draft["to"], draft["subject"], draft["body"])
        rows.append(
            "<tr>"
            f"<td>{draft['rank']}</td>"
            f"<td>{escape(draft['action_type'])}</td>"
            f"<td>{escape(', '.join(draft['to']))}</td>"
            f"<td>{escape(draft['subject'])}</td>"
            f"<td><a href=\"{escape(mailto_link)}\">Open email</a></td>"
            "</tr>"
        )
    rows.append("</tbody></table>")
    return "".join(rows)


def render_html_table(rows: list[dict], columns: list[tuple[str, str]]) -> str:
    if not rows:
        return "<p>No rows to display.</p>"

    parts = [
        "<table style=\"border-collapse:collapse;width:100%;font-family:Segoe UI, Arial, sans-serif;font-size:13px;\">",
        "<thead><tr>",
    ]
    for key, label in columns:
        parts.append(
            f"<th style=\"text-align:left;padding:8px;border-bottom:2px solid #d1d5db;background:#f8fafc;\">{escape(label)}</th>"
        )
    parts.append("</tr></thead><tbody>")

    for row in rows:
        parts.append("<tr>")
        for key, _ in columns:
            value = row.get(key, "")
            parts.append(
                f"<td style=\"vertical-align:top;padding:8px;border-bottom:1px solid #e5e7eb;\">{escape(str(value or ''))}</td>"
            )
        parts.append("</tr>")

    parts.append("</tbody></table>")
    return "".join(parts)


def normalize_name_tokens(value: str) -> set[str]:
    cleaned = []
    for char in (value or "").lower():
        cleaned.append(char if char.isalnum() else " ")
    tokens = {token for token in "".join(cleaned).split() if len(token) >= 3}
    stopwords = {
        "data", "model", "report", "semantic", "table", "view", "fact", "dim",
        "file", "sheet", "dump", "raw", "mart", "core", "base"
    }
    return {token for token in tokens if token not in stopwords}


def extract_business_entities(model: dict) -> set[str]:
    entities = set()
    entities |= normalize_name_tokens(model.get("display_name", ""))

    for source in model.get("data_sources", []):
        entities |= normalize_name_tokens(source.get("raw_type", ""))
        for key, value in source.get("normalized_details", {}).items():
            if key in {"path", "database", "url", "server", "site", "list", "table"}:
                entities |= normalize_name_tokens(str(value))

    sempy_metadata = model.get("sempy_metadata", {})
    for table in sempy_metadata.get("tables", []):
        entities |= normalize_name_tokens(str(table.get("Name") or table.get("name") or ""))

    for measure in sempy_metadata.get("measures", []):
        entities |= normalize_name_tokens(str(measure.get("Measure Name") or measure.get("name") or ""))

    for column in sempy_metadata.get("columns", []):
        entities |= normalize_name_tokens(
            str(
                column.get("Column Name")
                or column.get("name")
                or column.get("columnName")
                or ""
            )
        )

    return entities


def extract_column_tokens(model: dict) -> set[str]:
    tokens = set()
    for column in model.get("sempy_metadata", {}).get("columns", []):
        tokens |= normalize_name_tokens(
            str(
                column.get("Column Name")
                or column.get("name")
                or column.get("columnName")
                or ""
            )
        )
    return tokens


def detect_system_family(model: dict) -> str:
    source_text = " ".join(
        [
            source.get("raw_type", "")
            + " "
            + " ".join(f"{key}:{value}" for key, value in source.get("normalized_details", {}).items())
            for source in model.get("data_sources", [])
        ]
    ).lower()

    if "sap" in source_text:
        return "sap"
    if "sharepoint" in source_text or "sharepoint.com" in source_text or "excel" in source_text or "csv" in source_text:
        return "sharepoint_or_file"
    if "sql" in source_text or "synapse" in source_text or "warehouse" in source_text:
        return "sql_or_warehouse"
    if "dataverse" in source_text:
        return "dataverse"
    if "salesforce" in source_text:
        return "salesforce"
    return "other"


def jaccard_similarity(left: set[str], right: set[str]) -> float:
    if not left or not right:
        return 0.0
    intersection = left & right
    union = left | right
    return len(intersection) / len(union) if union else 0.0


def priority_rank(value: str) -> int:
    return {
        "high": 3,
        "medium": 2,
        "review": 1,
        "low": 0,
    }.get(str(value).strip().lower(), 0)


def is_likely_deployment_clone_candidate(candidate: dict) -> bool:
    impacted_models = {
        str(value).strip().lower()
        for value in (candidate.get("impacted_model_names") or [])
        if str(value).strip()
    }
    impacted_workspaces = {
        str(value).strip().lower()
        for value in (candidate.get("impacted_workspaces") or [])
        if str(value).strip()
    }

    return (
        len(impacted_models) == 1
        and len(impacted_workspaces) >= DEPLOYMENT_CLONE_MIN_WORKSPACES
    )


def detect_extract_signals(model: dict) -> list[str]:
    text = " ".join(
        [
            model.get("display_name", ""),
            model.get("description", "") or "",
            " ".join(
                [
                    source.get("raw_type", "") + " " + " ".join(source.get("normalized_details", {}).values())
                    for source in model.get("data_sources", [])
                ]
            ),
        ]
    ).lower()

    signals = []
    for token in ["extract", "export", "dump", "snapshot", "file", "excel", "csv", "sharepoint"]:
        if token in text:
            signals.append(token)
    return signals


def build_similarity_candidates(results: dict) -> list[dict]:
    models = results["semantic_models"]
    enriched_models = []
    for model in models:
        enriched_models.append(
            {
                "model": model,
                "entities": extract_business_entities(model),
                "columns": extract_column_tokens(model),
                "system_family": detect_system_family(model),
                "extract_signals": detect_extract_signals(model),
                "source_fingerprints": {source["fingerprint"] for source in model.get("data_sources", [])},
            }
        )

    candidates = []
    for index, left in enumerate(enriched_models):
        for right in enriched_models[index + 1:]:
            entity_similarity = jaccard_similarity(left["entities"], right["entities"])
            column_similarity = jaccard_similarity(left["columns"], right["columns"])
            overlap_score = round((entity_similarity * 0.65) + (column_similarity * 0.35), 3)

            if overlap_score < 0.35:
                continue

            same_source = bool(left["source_fingerprints"] & right["source_fingerprints"])
            if same_source:
                continue

            left_family = left["system_family"]
            right_family = right["system_family"]
            replacement_bias = {
                ("sharepoint_or_file", "sap"): "replace extracted SharePoint/file feed with governed SAP-based semantic source",
                ("sap", "sharepoint_or_file"): "replace extracted SharePoint/file feed with governed SAP-based semantic source",
                ("sharepoint_or_file", "sql_or_warehouse"): "replace spreadsheet/file feed with governed warehouse source",
                ("sql_or_warehouse", "sharepoint_or_file"): "replace spreadsheet/file feed with governed warehouse source",
            }.get((left_family, right_family))

            extract_bias = None
            if left["extract_signals"] and right_family in {"sap", "sql_or_warehouse", "dataverse", "salesforce"}:
                extract_bias = f"{left['model']['display_name']} looks extract-driven while {right['model']['display_name']} looks closer to a governed operational source"
            elif right["extract_signals"] and left_family in {"sap", "sql_or_warehouse", "dataverse", "salesforce"}:
                extract_bias = f"{right['model']['display_name']} looks extract-driven while {left['model']['display_name']} looks closer to a governed operational source"

            source_replacement_hypothesis = replacement_bias or extract_bias

            recommendation_type = (
                "source_replacement_candidate"
                if source_replacement_hypothesis
                else "semantic_overlap_candidate"
            )

            replacement_confidence = "high" if source_replacement_hypothesis and overlap_score >= 0.55 else (
                "medium" if source_replacement_hypothesis else "review"
            )

            candidates.append(
                {
                    "recommendation_type": recommendation_type,
                    "similarity_score": overlap_score,
                    "entity_similarity": round(entity_similarity, 3),
                    "column_similarity": round(column_similarity, 3),
                    "left_model_id": left["model"]["id"],
                    "left_model": left["model"]["display_name"],
                    "left_workspace": left["model"]["workspace_name"],
                    "left_datasource_count": len(left["model"].get("data_sources", [])),
                    "left_system_family": left_family,
                    "left_extract_signals": left["extract_signals"],
                    "right_model_id": right["model"]["id"],
                    "right_model": right["model"]["display_name"],
                    "right_workspace": right["model"]["workspace_name"],
                    "right_datasource_count": len(right["model"].get("data_sources", [])),
                    "right_system_family": right_family,
                    "right_extract_signals": right["extract_signals"],
                    "shared_entities": sorted(left["entities"] & right["entities"])[:20],
                    "shared_columns": sorted(left["columns"] & right["columns"])[:20],
                    "source_replacement_hypothesis": source_replacement_hypothesis,
                    "replacement_confidence": replacement_confidence,
                    "replacement_checklist": [
                        "Confirm both models represent the same business entity grain and refresh cadence.",
                        "Validate whether one source is a manual extract or downstream dump of the other.",
                        "Compare record counts, key columns, and refresh timestamps.",
                        "Identify the authoritative operational source and target shared semantic model.",
                        "Plan report migration away from the weaker source and retire duplicate logic.",
                    ],
                }
            )

    candidates.sort(key=lambda item: (-item["similarity_score"], item["left_model"], item["right_model"]))
    return candidates


def build_probable_duplicate_candidates(results: dict) -> list[dict]:
    model_lookup = {model["id"]: model for model in results.get("semantic_models", [])}
    pairwise_candidates = []

    for candidate in results.get("similarity_candidates", []):
        if candidate["recommendation_type"] != "semantic_overlap_candidate":
            continue
        left_model = model_lookup.get(candidate["left_model_id"])
        right_model = model_lookup.get(candidate["right_model_id"])
        if (
            left_model and is_excluded_semantic_model(left_model)
            or right_model and is_excluded_semantic_model(right_model)
        ):
            continue

        if candidate["similarity_score"] < PROBABLE_DUPLICATE_SIMILARITY_THRESHOLD:
            continue

        left_datasource_count = candidate.get("left_datasource_count", 0)
        right_datasource_count = candidate.get("right_datasource_count", 0)

        if left_datasource_count == 0 and right_datasource_count == 0:
            datasource_visibility = "missing_on_both_models"
        elif left_datasource_count == 0 or right_datasource_count == 0:
            datasource_visibility = "missing_on_one_model"
        else:
            datasource_visibility = "visible_on_both_models"

        if datasource_visibility == "visible_on_both_models":
            continue

        priority = "medium" if candidate["similarity_score"] >= 0.6 else "review"
        if datasource_visibility == "missing_on_both_models":
            why = (
                f"{candidate['left_model']} and {candidate['right_model']} have strong semantic overlap, "
                "but datasource metadata is missing on both models."
            )
        else:
            why = (
                f"{candidate['left_model']} and {candidate['right_model']} have strong semantic overlap, "
                "but datasource metadata is missing on one of the models."
            )

        pairwise_candidates.append(
            {
                "recommendation_type": "probable_duplicate_semantic_model",
                "priority": priority,
                "similarity_score": candidate["similarity_score"],
                "entity_similarity": candidate["entity_similarity"],
                "column_similarity": candidate["column_similarity"],
                "left_model_id": candidate["left_model_id"],
                "left_model": candidate["left_model"],
                "left_workspace": candidate["left_workspace"],
                "left_datasource_count": left_datasource_count,
                "right_model_id": candidate["right_model_id"],
                "right_model": candidate["right_model"],
                "right_workspace": candidate["right_workspace"],
                "right_datasource_count": right_datasource_count,
                "shared_entities": candidate["shared_entities"],
                "shared_columns": candidate["shared_columns"],
                "datasource_visibility": datasource_visibility,
                "why": why,
                "impacted_model_ids": [candidate["left_model_id"], candidate["right_model_id"]],
                "impacted_model_names": [candidate["left_model"], candidate["right_model"]],
                "impacted_workspaces": [candidate["left_workspace"], candidate["right_workspace"]],
            }
        )

    adjacency = defaultdict(set)
    edge_lookup = {}
    for candidate in pairwise_candidates:
        left_id = candidate["left_model_id"]
        right_id = candidate["right_model_id"]
        adjacency[left_id].add(right_id)
        adjacency[right_id].add(left_id)
        edge_lookup[frozenset({left_id, right_id})] = candidate

    grouped_candidates = []
    visited = set()
    for model_id in adjacency:
        if model_id in visited:
            continue

        stack = [model_id]
        component = set()
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.add(current)
            stack.extend(adjacency[current] - visited)

        component_edges = []
        component_ids = sorted(component)
        for index, left_id in enumerate(component_ids):
            for right_id in component_ids[index + 1:]:
                edge = edge_lookup.get(frozenset({left_id, right_id}))
                if edge:
                    component_edges.append(edge)

        if len(component_ids) == 2 and len(component_edges) == 1:
            grouped_candidates.append(component_edges[0])
            continue

        if not component_edges:
            continue

        model_names = sorted({model_lookup[item]["display_name"] for item in component_ids if item in model_lookup})
        workspaces = sorted({model_lookup[item]["workspace_name"] for item in component_ids if item in model_lookup})
        shared_entities = sorted(
            {
                entity
                for edge in component_edges
                for entity in edge.get("shared_entities", [])
            }
        )[:20]
        shared_columns = sorted(
            {
                column
                for edge in component_edges
                for column in edge.get("shared_columns", [])
            }
        )[:20]
        priority = max(component_edges, key=lambda item: priority_rank(item["priority"]))["priority"]
        if all(edge["datasource_visibility"] == "missing_on_both_models" for edge in component_edges):
            datasource_visibility = "missing_on_both_models"
            visibility_text = "all models in the cluster"
        else:
            datasource_visibility = "missing_on_one_model"
            visibility_text = "one or more models in the cluster"

        if len(model_names) == 1:
            cluster_label = model_names[0]
            why = (
                f"{cluster_label} appears as a highly similar semantic model across "
                f"{len(workspaces)} workspaces, but datasource metadata is missing on {visibility_text}."
            )
        else:
            cluster_label = f"{len(component_ids)} overlapping semantic models"
            why = (
                f"{len(component_ids)} semantic models form one overlap cluster across "
                f"{len(workspaces)} workspaces, but datasource metadata is missing on {visibility_text}."
            )

        grouped_candidates.append(
            {
                "recommendation_type": "probable_duplicate_semantic_model_cluster",
                "priority": priority,
                "similarity_score": round(
                    sum(edge["similarity_score"] for edge in component_edges) / len(component_edges),
                    3,
                ),
                "datasource_visibility": datasource_visibility,
                "model_name": cluster_label,
                "why": why,
                "shared_entities": shared_entities,
                "shared_columns": shared_columns,
                "impacted_model_ids": component_ids,
                "impacted_model_names": model_names,
                "impacted_workspaces": workspaces,
                "workspace_count": len(workspaces),
                "model_count": len(component_ids),
            }
        )

    grouped_candidates.sort(
        key=lambda item: (
            0 if item["datasource_visibility"] == "missing_on_both_models" else 1,
            -item["similarity_score"],
            item.get("model_name", item.get("left_model", "")),
            item.get("right_model", ""),
        )
    )
    return grouped_candidates


def build_executive_summary(results: dict) -> dict:
    candidates = results.get("similarity_candidates", [])
    replacement_candidates = [
        candidate for candidate in candidates if candidate["recommendation_type"] == "source_replacement_candidate"
    ]
    probable_duplicate_candidates = results.get("probable_duplicate_candidates", [])
    high_priority_duplicates = [
        recommendation for recommendation in results.get("recommendations", []) if recommendation["priority"] == "high"
    ]

    owner_contacts = defaultdict(lambda: {"models": set(), "workspaces": set()})
    for model in results.get("semantic_models", []):
        owner = model.get("configured_by")
        if owner:
            owner_contacts[owner]["models"].add(model["display_name"])
            owner_contacts[owner]["workspaces"].add(model["workspace_name"])

        for admin in model.get("workspace_admins", []):
            contact = admin.get("emailAddress") or admin.get("identifier")
            if contact:
                owner_contacts[contact]["models"].add(model["display_name"])
                owner_contacts[contact]["workspaces"].add(model["workspace_name"])

    top_contacts = sorted(
        [
            {
                "contact": contact,
                "model_count": len(payload["models"]),
                "workspace_count": len(payload["workspaces"]),
                "models": ", ".join(sorted(payload["models"])[:10]),
            }
            for contact, payload in owner_contacts.items()
        ],
        key=lambda item: (-item["model_count"], -item["workspace_count"], item["contact"]),
    )[:15]

    top_actions = []
    for candidate in replacement_candidates[:10]:
        top_actions.append(
            {
                "action_type": "source_replacement",
                "priority": candidate["replacement_confidence"],
                "summary": (
                    f"{candidate['left_model']} vs {candidate['right_model']} "
                    f"({candidate['source_replacement_hypothesis']})"
                ),
            }
        )

    for candidate in probable_duplicate_candidates[:10]:
        top_actions.append(
            {
                "action_type": "probable_duplicate_semantic_model",
                "priority": candidate["priority"],
                "summary": candidate["why"],
            }
        )

    for recommendation in high_priority_duplicates[:10]:
        top_actions.append(
            {
                "action_type": "duplicate_source_centralization",
                "priority": recommendation["priority"],
                "summary": recommendation["summary"],
            }
        )

    return {
        "headline_metrics": {
            "capacity_count": len(results.get("capacity_summary", [])),
            "workspace_count": len(results.get("workspace_summary", [])),
            "semantic_model_count": len(results.get("semantic_models", [])),
            "duplicate_source_group_count": len(results.get("duplicate_source_groups", [])),
            "probable_duplicate_semantic_model_count": len(probable_duplicate_candidates),
            "source_replacement_candidate_count": len(replacement_candidates),
            "access_issue_count": len(results.get("access_issues", [])),
        },
        "top_actions": top_actions[:15],
        "top_contacts": top_contacts,
    }


def build_network_payload(results: dict) -> dict:
    model_nodes = []
    source_nodes = []
    edges = []
    seen_sources = {}
    candidates = results.get("similarity_candidates", [])

    for model in results["semantic_models"]:
        model_nodes.append(
            {
                "id": f"model::{model['id']}",
                "label": model["display_name"],
                "type": "model",
                "workspace": model["workspace_name"],
            }
        )
        for source in model.get("data_sources", []):
            source_id = f"source::{source['fingerprint']}"
            if source_id not in seen_sources:
                seen_sources[source_id] = {
                    "id": source_id,
                    "label": source["raw_type"],
                    "type": "source",
                    "details": source["normalized_details"],
                }
                source_nodes.append(seen_sources[source_id])

            edges.append(
                {
                    "source": f"model::{model['id']}",
                    "target": source_id,
                    "edge_type": "uses_source",
                }
            )

    for candidate in candidates[:50]:
        edges.append(
            {
                "source": f"model::{candidate['left_model_id']}",
                "target": f"model::{candidate['right_model_id']}",
                "edge_type": candidate["recommendation_type"],
                "weight": candidate["similarity_score"],
            }
        )

    return {
        "nodes": model_nodes + source_nodes,
        "edges": edges,
    }


def render_network(results: dict) -> None:
    if not SHOW_NETWORK:
        return

    try:
        import math
        import plotly.graph_objects as go  # type: ignore
    except ImportError:
        print("plotly is not installed; skipping network view.")
        return

    network = results.get("network_graph", {})
    nodes = network.get("nodes", [])
    edges = network.get("edges", [])
    if not nodes:
        print("No network nodes found.")
        return

    positions = {}
    model_nodes = [node for node in nodes if node["type"] == "model"]
    source_nodes = [node for node in nodes if node["type"] == "source"]

    for idx, node in enumerate(model_nodes):
        angle = (2 * math.pi * idx / max(len(model_nodes), 1))
        positions[node["id"]] = (math.cos(angle) * 2.0, math.sin(angle) * 2.0)

    for idx, node in enumerate(source_nodes):
        angle = (2 * math.pi * idx / max(len(source_nodes), 1))
        positions[node["id"]] = (math.cos(angle) * 3.5, math.sin(angle) * 3.5)

    fig = go.Figure()

    edge_colors = {
        "uses_source": "#94a3b8",
        "source_replacement_candidate": "#ef4444",
        "semantic_overlap_candidate": "#f59e0b",
    }

    for edge in edges:
        source_pos = positions.get(edge["source"])
        target_pos = positions.get(edge["target"])
        if not source_pos or not target_pos:
            continue

        fig.add_trace(
            go.Scatter(
                x=[source_pos[0], target_pos[0]],
                y=[source_pos[1], target_pos[1]],
                mode="lines",
                line={
                    "width": 1 + float(edge.get("weight", 0.0)) * 4,
                    "color": edge_colors.get(edge["edge_type"], "#cbd5e1"),
                },
                hoverinfo="text",
                text=f"{edge['edge_type']}",
                showlegend=False,
            )
        )

    for node_type, color, size in [
        ("model", "#2563eb", 16),
        ("source", "#10b981", 12),
    ]:
        subset = [node for node in nodes if node["type"] == node_type]
        fig.add_trace(
            go.Scatter(
                x=[positions[node["id"]][0] for node in subset],
                y=[positions[node["id"]][1] for node in subset],
                mode="markers+text",
                marker={"size": size, "color": color},
                text=[node["label"] for node in subset],
                textposition="top center",
                hovertext=[
                    json.dumps(
                        to_json_safe({
                            key: value
                            for key, value in node.items()
                            if key not in {"id", "label", "type"}
                        }),
                        sort_keys=True,
                    )
                    for node in subset
                ],
                name=node_type,
            )
        )

    fig.update_layout(
        title="Semantic Model Consolidation Network",
        showlegend=True,
        xaxis={"visible": False},
        yaxis={"visible": False},
        height=800,
    )
    display(fig)


# ============================================================================
# ANALYZER
# ============================================================================

def analyze(
    capacity_name=None,
    include_sempy_metadata=True,
    duplicate_threshold=2,
    use_admin_apis=True,
) -> dict:
    auth_mode = AUTH_MODE.strip().lower()
    access_mode = ACCESS_MODE.strip().lower()
    fabric_token = get_access_token("fabric")
    powerbi_token = get_access_token("powerbi")
    scan_datasource_fallback_count = 0

    fabric_base_url = "https://api.fabric.microsoft.com/v1"
    fabric_capacities = []
    try:
        fabric_capacities = get_paginated(f"{fabric_base_url}/capacities", fabric_token, "value")
    except Exception:
        fabric_capacities = []

    use_admin_path = use_admin_apis and access_mode in {"auto", "capacity_admin"}
    admin_fallback_reason = None

    try:
        capacities = list_admin_capacities(powerbi_token) if use_admin_path else fabric_capacities
    except Exception as exc:
        if access_mode == "capacity_admin":
            raise
        use_admin_path = False
        capacities = fabric_capacities
        admin_fallback_reason = f"Admin API capacity discovery unavailable with current user context: {exc}"

    selected_capacity_ids = {
        item["id"]
        for item in capacities
        if capacity_name is None
        or item.get("displayName", item.get("displayName", "")).lower() == capacity_name.lower()
    }

    if use_admin_path:
        try:
            filtered_workspaces = []
            if selected_capacity_ids:
                for selected_capacity_id in selected_capacity_ids:
                    filtered_workspaces.extend(list_admin_workspaces(fabric_token, selected_capacity_id))
            elif capacity_name is None:
                filtered_workspaces = list_admin_workspaces(fabric_token, None)

            if not filtered_workspaces:
                if selected_capacity_ids:
                    for selected_capacity_id in selected_capacity_ids:
                        filtered_workspaces.extend(list_powerbi_admin_groups(powerbi_token, selected_capacity_id))
                elif capacity_name is None:
                    filtered_workspaces = list_powerbi_admin_groups(powerbi_token, None)
        except Exception as exc:
            if access_mode == "capacity_admin":
                raise
            use_admin_path = False
            admin_fallback_reason = f"Admin workspace discovery unavailable with current user context: {exc}"

    if not use_admin_path:
        workspaces = get_paginated(f"{fabric_base_url}/workspaces", fabric_token, "value")
        filtered_workspaces = [
            workspace
            for workspace in workspaces
            if capacity_name is None or workspace.get("capacityId") in selected_capacity_ids
        ]
        filtered_workspaces = filter_workspaces_for_user_scope(filtered_workspaces)

    workspace_map = {workspace["id"]: workspace for workspace in filtered_workspaces}
    scan_workspace_lookup = {}
    datasource_instance_lookup = {}

    if use_admin_path and USE_WORKSPACE_SCAN_API and workspace_map:
        try:
            scan_payload = scan_workspaces_metadata(powerbi_token, list(workspace_map.keys()))
            scan_workspace_lookup = {
                workspace["id"]: workspace
                for workspace in scan_payload.get("workspaces", [])
            }
            datasource_instance_lookup = {
                item.get("datasourceInstanceId") or item.get("datasourceId") or item.get("id"): item
                for item in (
                    scan_payload.get("datasourceInstances", [])
                    + scan_payload.get("misconfiguredDatasourceInstances", [])
                )
                if item.get("datasourceInstanceId") or item.get("datasourceId") or item.get("id")
            }
            dataset_records = []
            for workspace_id, workspace_payload in scan_workspace_lookup.items():
                for dataset in workspace_payload.get("datasets", []):
                    dataset["workspaceId"] = workspace_id
                    dataset_records.append(dataset)
        except Exception as exc:
            if access_mode == "capacity_admin":
                raise
            use_admin_path = False
            admin_fallback_reason = f"Metadata scan API unavailable with current user context: {exc}"

    if use_admin_path and not scan_workspace_lookup:
        dataset_records = [
            dataset
            for dataset in list_admin_datasets(powerbi_token)
            if dataset.get("workspaceId") in workspace_map
        ]
    elif not use_admin_path:
        dataset_records = []
        for workspace in filtered_workspaces:
            workspace_id = workspace["id"]
            models = get_paginated(
                f"{fabric_base_url}/workspaces/{workspace_id}/semanticModels",
                fabric_token,
                "value",
            )
            for model in models:
                dataset_records.append(
                    {
                        "id": model["id"],
                        "displayName": model.get("displayName", model["id"]),
                        "description": model.get("description"),
                        "workspaceId": workspace_id,
                    }
                )

    semantic_models = []
    duplicate_map = defaultdict(list)
    fingerprint_details = {}
    access_issues = []
    workspace_users_cache = {}

    for model in dataset_records:
        workspace_id = model["workspaceId"]
        workspace = workspace_map[workspace_id]
        workspace_name = workspace.get("displayName") or workspace.get("name") or workspace_id
        model_id = model.get("id") or model.get("objectId")
        model_name = model.get("displayName", model.get("name", model_id))
        configured_by = model.get("configuredBy")

        datasources = []
        if use_admin_path and USE_WORKSPACE_SCAN_API:
            datasources = get_scan_linked_datasources(model, datasource_instance_lookup)
            if (
                not datasources
                and ENRICH_SCAN_MODELS_WITH_DIRECT_DATASOURCE_LOOKUP
                and model_id
            ):
                try:
                    datasources = get_dataset_datasources(
                        powerbi_token=powerbi_token,
                        workspace_id=workspace_id,
                        dataset_id=model_id,
                        use_admin_apis=True,
                    )
                    if datasources:
                        scan_datasource_fallback_count += 1
                except Exception as exc:
                    access_issues.append(
                        {
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "semantic_model_id": model_id,
                            "semantic_model_name": model_name,
                            "stage": "get_dataset_datasources_after_scan_gap",
                            "error": str(exc),
                        }
                    )
        else:
            try:
                datasources = get_dataset_datasources(
                    powerbi_token=powerbi_token,
                    workspace_id=workspace_id,
                    dataset_id=model_id,
                    use_admin_apis=use_admin_path,
                )
            except Exception as exc:
                access_issues.append(
                    {
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "semantic_model_id": model_id,
                        "semantic_model_name": model_name,
                        "stage": (
                            "get_dataset_datasources_as_admin"
                            if use_admin_path
                            else "get_dataset_datasources"
                        ),
                        "error": str(exc),
                    }
                )

        workspace_admins = []
        if INCLUDE_RESPONSIBILITY:
            if use_admin_path and USE_WORKSPACE_SCAN_API and workspace_id in scan_workspace_lookup:
                workspace_admins = [
                    user
                    for user in scan_workspace_lookup[workspace_id].get("users", [])
                    if str(user.get("groupUserAccessRight", "")).lower() == "admin"
                ]
            else:
                try:
                    if workspace_id not in workspace_users_cache:
                        workspace_users_cache[workspace_id] = get_group_users_as_admin(powerbi_token, workspace_id)
                    workspace_admins = [
                        user
                        for user in workspace_users_cache[workspace_id]
                        if str(user.get("groupUserAccessRight", "")).lower() == "admin"
                    ]
                except Exception as exc:
                    access_issues.append(
                        {
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "semantic_model_id": model_id,
                            "semantic_model_name": model_name,
                            "stage": "get_group_users_as_admin",
                            "error": str(exc),
                        }
                    )

        normalized_sources = []
        unique_fingerprints = {}

        for datasource in datasources:
            fingerprint, normalized = normalize_datasource(
                datasource.get("datasourceType", "unknown"),
                datasource.get("connectionDetails", {}),
            )
            source_info = {
                "raw_type": datasource.get("datasourceType", "unknown"),
                "fingerprint": fingerprint,
                "normalized_details": normalized,
            }
            normalized_sources.append(source_info)
            unique_fingerprints[fingerprint] = source_info

        model_info = {
            "id": model_id,
            "display_name": model_name,
            "workspace_id": workspace_id,
            "workspace_name": workspace_name,
            "capacity_id": workspace.get("capacityId"),
            "description": model.get("description"),
            "configured_by": configured_by,
            "workspace_admins": workspace_admins,
            "data_sources": normalized_sources,
            "sempy_metadata": (
                {
                    "table_count": len(model.get("tables", [])),
                    "tables": model.get("tables", []),
                    "measure_count": sum(len(table.get("measures", [])) for table in model.get("tables", [])),
                    "column_count": sum(len(table.get("columns", [])) for table in model.get("tables", [])),
                    "columns": [
                        column
                        for table in model.get("tables", [])
                        for column in table.get("columns", [])
                    ],
                }
                if use_admin_path and USE_WORKSPACE_SCAN_API and model.get("tables") is not None
                else (
                    try_get_semantic_model_metadata(model_name, workspace_name)
                    if include_sempy_metadata
                    else {}
                )
            ),
        }
        semantic_models.append(model_info)

        for fingerprint, source_info in unique_fingerprints.items():
            duplicate_map[fingerprint].append(model_info)
            fingerprint_details[fingerprint] = {
                "raw_type": source_info["raw_type"],
                "normalized_details": source_info["normalized_details"],
            }

    duplicate_groups = []
    for fingerprint, models in duplicate_map.items():
        included_models = [model for model in models if not is_excluded_semantic_model(model)]
        if len(included_models) < duplicate_threshold:
            continue

        if len(models) < duplicate_threshold:
            continue

        duplicate_groups.append(
            {
                "fingerprint": fingerprint,
                "datasource_type": fingerprint_details[fingerprint]["raw_type"],
                "normalized_details": fingerprint_details[fingerprint]["normalized_details"],
                "model_count": len(included_models),
                "models": [
                    {
                        "id": item["id"],
                        "display_name": item["display_name"],
                        "workspace_id": item["workspace_id"],
                        "workspace_name": item["workspace_name"],
                        "capacity_id": item["capacity_id"],
                    }
                    for item in sorted(
                        included_models,
                        key=lambda value: (value["workspace_name"], value["display_name"]),
                    )
                ],
            }
        )

    duplicate_groups.sort(key=lambda item: (-item["model_count"], item["datasource_type"], item["fingerprint"]))

    recommendations = []
    for group in duplicate_groups:
        model_count = group["model_count"]
        if model_count >= 4:
            priority = "high"
        elif model_count >= 3:
            priority = "medium"
        else:
            priority = "low"

        recommendations.append(
            {
                "priority": priority,
                "title": f"Centralize duplicated {group['datasource_type']} source into a shared semantic layer",
                "summary": (
                    f"This source appears in {model_count} semantic models. "
                    "Consider moving shared entities and core business logic into a centralized semantic model."
                ),
                "datasource": group["normalized_details"],
                "impacted_models": [model["display_name"] for model in group["models"]],
            }
        )

    results = {
        "capacity_summary": [
            item for item in capacities if capacity_name is None or item["id"] in selected_capacity_ids
        ],
        "workspace_summary": [
            {
                "id": workspace["id"],
                "display_name": workspace.get("displayName") or workspace.get("name") or workspace["id"],
                "capacity_id": workspace.get("capacityId"),
            }
            for workspace in filtered_workspaces
        ],
        "semantic_models": semantic_models,
        "duplicate_source_groups": duplicate_groups,
        "centralization_candidates": duplicate_groups,
        "recommendations": recommendations,
        "access_issues": access_issues,
        "scan_mode": "admin_api" if use_admin_path else "workspace_api",
        "generated_at_epoch": int(time.time()),
        "diagnostics": (
            {
                "auth_mode": AUTH_MODE,
                "access_mode": ACCESS_MODE,
                "use_admin_apis_requested": use_admin_apis,
                "use_admin_apis_effective": use_admin_path,
                "admin_fallback_reason": admin_fallback_reason,
                "scan_datasource_fallback_count": scan_datasource_fallback_count,
                "fabric_capacity_count": len(fabric_capacities),
                "admin_capacity_count": len(capacities) if use_admin_path else None,
                "selected_capacity_ids": sorted(selected_capacity_ids),
                "capacity_names": [
                    {
                        "id": item["id"],
                        "display_name": item.get("displayName", item["id"]),
                    }
                    for item in capacities
                ],
                "workspace_count": len(filtered_workspaces),
                "semantic_models_without_datasources": sum(
                    1 for model in semantic_models if not model.get("data_sources")
                ),
                "excluded_semantic_model_ids": sorted(
                    str(value).strip()
                    for value in EXCLUDED_SEMANTIC_MODEL_IDS
                    if str(value).strip()
                ),
                "excluded_semantic_model_names": sorted(normalize_string_set(EXCLUDED_SEMANTIC_MODEL_NAMES)),
                "excluded_workspace_names": sorted(normalize_string_set(EXCLUDED_WORKSPACE_NAMES)),
                "excluded_model_workspace_pairs": sorted(
                    [list(item) for item in normalize_model_workspace_pairs(EXCLUDED_MODEL_WORKSPACE_PAIRS)]
                ),
                "target_workspace_ids": TARGET_WORKSPACE_IDS,
                "target_workspace_names": TARGET_WORKSPACE_NAMES,
                "workspace_samples": [
                    {
                        "id": workspace["id"],
                        "display_name": workspace.get("displayName") or workspace.get("name") or workspace["id"],
                        "capacity_id": workspace.get("capacityId"),
                    }
                    for workspace in filtered_workspaces[:10]
                ],
            }
            if ENABLE_DIAGNOSTICS
            else {}
        ),
    }

    results["similarity_candidates"] = build_similarity_candidates(results)
    results["probable_duplicate_candidates"] = build_probable_duplicate_candidates(results)
    results["visual_tables"] = build_visual_tables(results)
    results["visual_tables"]["source_replacement_table"] = [
        candidate
        for candidate in results["similarity_candidates"]
        if candidate["recommendation_type"] == "source_replacement_candidate"
    ]
    results["network_graph"] = build_network_payload(results)
    results["executive_summary"] = build_executive_summary(results)
    results["action_plan"] = build_action_plan(results)
    if results.get("diagnostics") is not None:
        results["diagnostics"]["suppressed_probable_duplicate_count"] = results.get(
            "suppressed_probable_duplicate_count",
            0,
        )
    results["action_plan_text"] = build_action_plan_text(results)
    results["email_drafts"] = build_email_drafts(results)
    results["condensed_results"] = build_condensed_results(results)
    return results


def display_results(results: dict) -> None:
    can_show_tables = SHOW_TABLES
    pd = None
    html_display = None
    try:
        import pandas as pd  # type: ignore
        from IPython.display import HTML as NotebookHTML  # type: ignore
        html_display = NotebookHTML
    except ImportError:
        if SHOW_TABLES:
            print("pandas is not installed; skipping dataframe views.")
        can_show_tables = False

    compact_action_plan_rows = results.get("visual_tables", {}).get("action_plan_table", [])
    consolidation_rows = results.get("visual_tables", {}).get("consolidation_candidates_table", [])
    responsibility_rows = results.get("visual_tables", {}).get("responsibility_table", [])
    probable_duplicate_rows = results.get("visual_tables", {}).get("probable_duplicate_table", [])
    replacement_rows = results.get("visual_tables", {}).get("source_replacement_table", [])
    executive_summary = results.get("executive_summary", {})
    action_plan_rows = results.get("action_plan", [])
    action_plan_text = results.get("action_plan_text", "")
    email_drafts = results.get("email_drafts", [])
    duplicate_groups = results.get("duplicate_source_groups", [])
    diagnostics = results.get("diagnostics", {})

    print("\nRun Diagnostics")
    diagnostic_summary = {
        "scan_mode": results.get("scan_mode"),
        "semantic_model_count": len(results.get("semantic_models", [])),
        "duplicate_source_group_count": len(duplicate_groups),
        "semantic_models_without_datasources": diagnostics.get("semantic_models_without_datasources"),
        "suppressed_probable_duplicate_count": diagnostics.get("suppressed_probable_duplicate_count"),
        "scan_datasource_fallback_count": diagnostics.get("scan_datasource_fallback_count"),
        "use_admin_apis_effective": diagnostics.get("use_admin_apis_effective"),
        "admin_fallback_reason": diagnostics.get("admin_fallback_reason"),
    }
    print(json.dumps(to_json_safe(diagnostic_summary), indent=2))

    if can_show_tables and SHOW_ONLY_ACTIONABLE and compact_action_plan_rows:
        print("\nAction Plan")
        if html_display is not None:
            display(
                html_display(
                    render_html_table(
                        compact_action_plan_rows,
                        [
                            ("rank", "Rank"),
                            ("priority", "Priority"),
                            ("action_type", "Action"),
                            ("impacted_model_count", "Models"),
                            ("workspace_count", "Workspaces"),
                            ("next_step", "Next Step"),
                        ],
                    )
                )
            )
        else:
            display(pd.DataFrame(compact_action_plan_rows))
    elif SHOW_ONLY_ACTIONABLE and action_plan_text:
        print("\nAction Plan Narrative")
        print(action_plan_text)

    if SHOW_EXECUTIVE_SUMMARY:
        print("\nExecutive Summary")
        headline_metrics = executive_summary.get("headline_metrics", {})
        if can_show_tables and headline_metrics:
            display(pd.DataFrame([headline_metrics]))
        elif headline_metrics:
            print(json.dumps(to_json_safe(headline_metrics), indent=2))

        top_actions = executive_summary.get("top_actions", [])
        if can_show_tables and top_actions:
            print("\nTop Recommended Actions")
            display(pd.DataFrame(top_actions))
        elif top_actions:
            print("\nTop Recommended Actions")
            print(json.dumps(to_json_safe(top_actions), indent=2))

        top_contacts = executive_summary.get("top_contacts", [])
        if can_show_tables and top_contacts:
            print("\nTop Contacts")
            display(pd.DataFrame(top_contacts))
        elif top_contacts:
            print("\nTop Contacts")
            print(json.dumps(to_json_safe(top_contacts), indent=2))

    print("\nConsolidation Candidates")
    if can_show_tables and consolidation_rows and not SHOW_ONLY_ACTIONABLE:
        display(pd.DataFrame(consolidation_rows))
    elif not consolidation_rows:
        print("No duplicate-source groups were identified in this run.")
    else:
        print("Detailed consolidation table hidden; use Action Plan above.")

    print("\nResponsibility View")
    if responsibility_rows:
        if SHOW_ONLY_ACTIONABLE:
            actionable_ids = set(results.get("actionable_model_ids", []))
            actionable_names = {
                model["display_name"]
                for model in results.get("semantic_models", [])
                if model["id"] in actionable_ids
            }
            filtered_rows = [
                row for row in responsibility_rows if row["semantic_model"] in actionable_names
            ]
            if can_show_tables and filtered_rows:
                display(pd.DataFrame(filtered_rows))
            elif filtered_rows:
                print(json.dumps(to_json_safe(filtered_rows), indent=2))
            else:
                print("No actionable responsibility rows found.")
        elif can_show_tables:
            display(pd.DataFrame(responsibility_rows))
        else:
            print(json.dumps(to_json_safe(responsibility_rows), indent=2))
    else:
        print("No responsibility rows found.")

    print("\nSource Replacement Candidates")
    if can_show_tables and replacement_rows:
        if html_display is not None:
            display(
                html_display(
                    render_html_table(
                        replacement_rows,
                        [
                            ("priority", "Priority"),
                            ("similarity_score", "Score"),
                            ("left_model", "Model A"),
                            ("left_workspace", "Workspace A"),
                            ("right_model", "Model B"),
                            ("right_workspace", "Workspace B"),
                            ("hypothesis", "Hypothesis"),
                        ],
                    )
                )
            )
        else:
            display(pd.DataFrame(replacement_rows))
    elif replacement_rows:
        print(json.dumps(to_json_safe(replacement_rows), indent=2))
    else:
        print("No source replacement candidates were identified in this run.")

    print("\nProbable Duplicate Semantic Models")
    if can_show_tables and probable_duplicate_rows:
        if html_display is not None:
            display(
                html_display(
                    render_html_table(
                        probable_duplicate_rows,
                        [
                            ("priority", "Priority"),
                            ("similarity_score", "Score"),
                            ("model_scope", "Model Scope"),
                            ("model_count", "Models"),
                            ("workspace_count", "Workspaces"),
                            ("workspaces", "Workspace List"),
                            ("datasource_visibility", "Datasource Visibility"),
                            ("why", "Reason"),
                        ],
                    )
                )
            )
        else:
            display(pd.DataFrame(probable_duplicate_rows))
    elif probable_duplicate_rows:
        print(json.dumps(to_json_safe(probable_duplicate_rows), indent=2))
    elif not probable_duplicate_rows:
        print("No probable duplicate semantic-model groups were identified in this run.")

    if DISPLAY_EMAIL_DRAFTS and email_drafts:
        print("\nEmail Drafts")
        if can_show_tables and html_display is not None:
            display(html_display(render_email_drafts_table(email_drafts)))
        else:
            for draft in email_drafts:
                print(f"\nDraft {draft['rank']} - {draft['subject']}")
                print(f"To: {', '.join(draft['to'])}")
                print(build_mailto_link(draft["to"], draft["subject"], draft["body"]))
    elif GENERATE_EMAIL_DRAFTS and DISPLAY_EMAIL_DRAFTS:
        print("\nEmail Drafts")
        if not action_plan_rows:
            print("No drafts were generated because no actionable consolidation items were identified.")
        else:
            print(
                "No drafts were generated because the actionable items did not include any "
                "dataset owners or workspace admin email addresses."
            )

    render_network(results)

In [ ]:
# %% Cell 5 - Run Analyzer

# ============================================================================
# RUN
# ============================================================================

results = analyze(
    capacity_name=CAPACITY_NAME,
    include_sempy_metadata=INCLUDE_SEMPY_METADATA,
    duplicate_threshold=DUPLICATE_THRESHOLD,
    use_admin_apis=USE_ADMIN_APIS,
)

json_payload = results.get("condensed_results", results) if SHOW_ONLY_ACTIONABLE else results
if PRINT_FULL_JSON:
    json_payload = results

if not VISUALS_ONLY:
    print(json.dumps(to_json_safe(json_payload), indent=2))

display_results(results)
